# Microsoft Word (2026년 최신 권장 사용법)

[Microsoft Word](https://www.microsoft.com/en-us/microsoft-365/word)는 Microsoft 에서 개발한 워드 프로세서입니다.

이 노트북은 `Word`(.docx) 문서를 후속 처리(분할·임베딩)에 사용할 수 있는 `Document` 형식으로 로드하는 방법을 다룹니다.

> **⚠️ 2026년 9월 기준 변경 사항 (공통)**
>
> - 책에서 사용한 `langchain_community.document_loaders` 는 **`langchain-community` 패키지 sunset**(2026년 5월 발표, 6월 저장소 아카이브)으로 더 이상 유지보수되지 않습니다. 설치·import 는 여전히 되지만 새 프로젝트의 기반으로 권장되지 않습니다.
> - LangChain 의 현재 방향은 ① **전용 통합 패키지**(`langchain-upstage`, `langchain-unstructured`, `langchain-pymupdf4llm`, `langchain-docling` 등)를 쓰거나, ② 파싱 라이브러리를 **직접 사용**하고 결과를 `langchain_core.documents.Document` 로 감싸는 것(필요하면 `BaseLoader` 를 상속한 작은 로더 클래스 작성)입니다.
> - `Document`, `BaseLoader`, 텍스트 분할기(`langchain-text-splitters`)는 그대로 유지되는 핵심 인터페이스입니다.

**이 노트북에서 바뀐 점**

| 책(구버전, `langchain_community`) | 현재 권장 |
|---|---|
| `Docx2txtLoader` (`docx2txt`) | **`python-docx`** 직접 사용 — 문단/표/제목 스타일까지 읽을 수 있음 (`docx2txt` 는 오래 업데이트되지 않음) |
| `UnstructuredWordDocumentLoader` | **`langchain-unstructured`** 의 `UnstructuredLoader` |
| `mode="elements"` | `UnstructuredLoader` 는 기본이 요소 단위 / `python-docx` 로도 요소 단위 로더 구현 |
| (신규) | **`langchain-docling`** 의 `DoclingLoader` (Markdown 변환) |

In [ ]:
# 설치
# !pip install -qU langchain-core python-docx
# !pip install -qU langchain-unstructured "unstructured[docx]"   # UnstructuredLoader 사용 시
# !pip install -qU langchain-docling                             # DoclingLoader 사용 시

## python-docx 로 직접 로드 (구 `Docx2txtLoader`)

`Document.iter_inner_content()` 는 본문의 **문단과 표를 문서 순서대로** 돌려줍니다. 표는 Markdown 표로 바꿔서 본문에 포함시킵니다.

In [ ]:
from pathlib import Path
from typing import Iterator, Literal

import docx  # python-docx
from docx.table import Table
from docx.text.paragraph import Paragraph
from langchain_core.document_loaders import BaseLoader
from langchain_core.documents import Document


def table_to_markdown(table: Table) -> str:
    rows = [[cell.text.strip().replace("\n", " ") for cell in row.cells] for row in table.rows]
    if not rows:
        return ""
    header, *body = rows
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(r) + " |" for r in body]
    return "\n".join(lines)


def paragraph_category(p: Paragraph) -> str:
    style = (p.style.name if p.style is not None else "") or ""
    if style == "Title" or style.startswith("Heading"):
        return "Title"
    if "List" in style:
        return "ListItem"
    return "NarrativeText"


class DocxLoader(BaseLoader):
    """python-docx 기반 Word 로더 (single / elements 모드)"""

    def __init__(self, file_path: str | Path, mode: Literal["single", "elements"] = "single") -> None:
        self.file_path = Path(file_path)
        self.mode = mode

    def _elements(self) -> Iterator[tuple[str, str]]:
        document = docx.Document(self.file_path)
        for block in document.iter_inner_content():
            if isinstance(block, Paragraph):
                if block.text.strip():
                    yield paragraph_category(block), block.text.strip()
            elif isinstance(block, Table):
                yield "Table", table_to_markdown(block)

    def lazy_load(self) -> Iterator[Document]:
        source = str(self.file_path)
        if self.mode == "single":
            text = "\n\n".join(content for _, content in self._elements())
            yield Document(page_content=text, metadata={"source": source})
            return
        for i, (category, content) in enumerate(self._elements()):
            yield Document(
                page_content=content,
                metadata={"source": source, "category": category, "element_index": i},
            )

In [ ]:
loader = DocxLoader("./data/sample-word-document.docx")  # 문서 로더 초기화

docs = loader.load()  # 문서 로딩

print(len(docs))

In [ ]:
print(docs[0].page_content[:500])

## UnstructuredLoader (구 `UnstructuredWordDocumentLoader`)

`UnstructuredLoader` 는 파일 형식을 자동 감지하며, 결과는 **요소(element) 단위**의 여러 `Document` 로 반환됩니다.

In [ ]:
from langchain_unstructured import UnstructuredLoader

# 비구조화된 워드 문서 로더 인스턴스화
loader = UnstructuredLoader("./data/sample-word-document.docx")

# 문서 로드
docs = loader.load()

print(len(docs))

책에서처럼 1개의 단일 Document 가 필요하면 요소들을 합칩니다.

In [ ]:
single_doc = Document(
    page_content="\n\n".join(d.page_content for d in docs),
    metadata={"source": "./data/sample-word-document.docx"},
)
print(single_doc.metadata)
print(len(single_doc.page_content))

내부적으로 Unstructured 는 텍스트 덩어리마다 서로 다른 "**요소**"를 만듭니다. 각 요소의 종류는 `metadata["category"]` 로 확인할 수 있습니다.

In [ ]:
# 첫번째 요소의 내용 출력
print(docs[0].page_content)

In [ ]:
# 첫번째 요소의 메타데이터 출력
docs[0].metadata

In [ ]:
from collections import Counter

Counter(d.metadata.get("category") for d in docs)

같은 요소 단위 결과를 `python-docx` 로도 얻을 수 있습니다. (의존성이 가볍고 빠름)

In [ ]:
element_docs = DocxLoader("./data/sample-word-document.docx", mode="elements").load()
print(len(element_docs))
element_docs[0]

## (추가) DoclingLoader — Markdown 으로 변환

제목 계층·표 구조를 Markdown 으로 보존하므로, 이후 `MarkdownHeaderTextSplitter` 로 섹션 단위 분할하기 좋습니다.

In [ ]:
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

docs = DoclingLoader(
    file_path="./data/sample-word-document.docx",
    export_type=ExportType.MARKDOWN,
).load()
print(docs[0].page_content[:500])